In [ ]:
import copy
from pathlib import Path
import sys

repo_root = Path.cwd()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.insert(0, str(repo_root / "src"))

from models import get_classification_model

convnext_model = get_classification_model("convnext_tiny", pretrained=True, num_classes=37)
swin_transformer_model = get_classification_model("swin_tiny_patch4_window7_224", pretrained=True, num_classes=37)
resnet50_model = get_classification_model("resnet50", pretrained=True, num_classes=37)

initial_model_states = {
    "convnext_tiny": copy.deepcopy(convnext_model.state_dict()),
    "swin_tiny": copy.deepcopy(swin_transformer_model.state_dict()),
    "resnet50": copy.deepcopy(resnet50_model.state_dict()),
}

# Sample image inference time

In [ ]:
import torch
from torch.utils.flop_counter import FlopCounterMode

from benchmark import inference_time

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

NUM_ITERATIONS = 100
BATCH_SIZE = 1
IMAGE_SIZE = (224, 224)

input_tensor = torch.randn(BATCH_SIZE, 3, *IMAGE_SIZE)
input_tensor = input_tensor.to(DEVICE)

models = {
    "convnext_tiny": convnext_model,
    "swin_tiny": swin_transformer_model,
    "resnet50": resnet50_model,
}

for model_name, model in models.items():
    model.load_state_dict(initial_model_states[model_name])
    avg_time_per_iteration, total_flops = inference_time(model, input_tensor, device=DEVICE, num_iterations=NUM_ITERATIONS)

    print(f"Average inference time for {model_name}: {avg_time_per_iteration:.6f} seconds per iteration")
    print(f"Total FLOPs for {model_name}: {total_flops}")

Average inference time for convnext_tiny: 0.002124 seconds per iteration
Average inference time for swin_tiny: 0.003529 seconds per iteration
Average inference time for resnet50: 0.003363 seconds per iteration


# Sample batch inference time

In [ ]:
import torch
import time

from benchmark import inference_time

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)

NUM_ITERATIONS = 100
BATCH_SIZE = 32
IMAGE_SIZE = (224, 224)

input_tensor = torch.randn(BATCH_SIZE, 3, *IMAGE_SIZE)
input_tensor = input_tensor.to(DEVICE)

models = {
    "convnext_tiny": convnext_model,
    "swin_tiny": swin_transformer_model,
    "resnet50": resnet50_model,
}

for model_name, model in models.items():
    model.load_state_dict(initial_model_states[model_name])
    avg_time_per_iteration, total_flops = inference_time(model, input_tensor, device=DEVICE, num_iterations=NUM_ITERATIONS)

    print(f"Average inference time for {model_name} with batch size {BATCH_SIZE}: {avg_time_per_iteration:.6f} seconds per iteration")
    print(f"Total FLOPs for {model_name}: {total_flops}")

Average inference time for convnext_tiny with batch size 32: 0.068509 seconds per iteration
Average inference time for swin_tiny with batch size 32: 0.093636 seconds per iteration
Average inference time for resnet50 with batch size 32: 0.066126 seconds per iteration
